In [1]:
import json
from collections import defaultdict
from pathlib import Path

import pytest
from astropy.coordinates import SkyCoord
from astropy import units as u
from astroquery.alma import Alma
from dl import queryClient as qc


SEARCH_RADIUS_ARCSEC = 60.0
MAX_RESOLUTION_ARCSEC = 1.0
BANDS = [6,7]

In [ ]:
def flatten(obj, prefix=""):
    """Flattens a dictionary or list into a list of key-value pairs"""
    results = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            results.extend(flatten(v, f"{prefix}.{k}" if prefix else k))
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            results.extend(flatten(v, f"{prefix}[{i}]"))
    else:
        results.append((prefix, obj))
    return results


def test_output_is_valid_json(agent_output):
    """
    Test that the agent's output.json file is valid JSON (the agent_output fixture will raise an error if the file isn't)
    """
    assert isinstance(agent_output, (dict, list))


def test_reports_a_plausible_observation_count(flat, reference_groups):
    
    

### Fixtures ==> vars

In [ ]:

output_path = Path("test_output.json")
agent_output = json.loads(output_path.read_text())
flat =  list(flatten(agent_output))
reference_coord = SkyCoord.from_name("HH 212")

alma = Alma()
tbl = alma.query_region(
    reference_coord, radius=SEARCH_RADIUS_ARCSEC * u.arcsec,
    band_list=BANDS, public=True, science=True,
)
tbl = tbl[tbl["spatial_resolution"] < MAX_RESOLUTION_ARCSEC]
groups = defaultdict(list)
for row in tbl:
    groups[str(row["member_ous_uid"])].append(row)
reference_groups = groups


### NEW Funcs

In [ ]:
def _find_observation_arrays(obj, path=""):
    """
    Yield (path, list) for every list of dicts in the structure where the
    dicts look like they represent individual observations
    
    NOTE: This function was written using Claude Sonnet 5, with my comments added"""
    
    OBSERVATION_LIKE_KEYS = {"project", "proposal", "band", "resolution", "uid", "mous"}

    if isinstance(obj, dict):
        #If dict, recursively search its values
        for k, v in obj.items():
            yield from _find_observation_arrays(v, f"{path}.{k}" if path else k)
    elif isinstance(obj, list):
        # If list, check if it is a list of dicts...
        if obj and all(isinstance(item, dict) for item in obj):
            sample_keys = " ".join(str(k).lower() for item in obj for k in item.keys())
            # If any of the keys in the dicts match known observation-like keys, yield this list
            if any(term in sample_keys for term in OBSERVATION_LIKE_KEYS):
                yield path, obj
        for i, v in enumerate(obj):
            # Recursively search each item in the list
            yield from _find_observation_arrays(v, f"{path}[{i}]")


def test_observation_count(agent_output, flat, reference_groups):
    expected = len(reference_groups)
    tolerance = max(1, round(0.15 * expected))

    #Level 1:
    #Start by searching for any keys in the output that contain potential observation count identifiers
    potential_identifiers = [
        'obs_count', 'observation_count', 'num_observations',
        'number_of_observations', 'num_obs', 'total_observations',
        'observations_total', 'observations_count', 'obs_total',
        'obs_num', 'obs_number', 'obs_count_total', 'obs_count_num',
        'obs_count_number',
    ]    
    #Check for any keys in the output that contain any of the potential identifiers
    obs_count = [
        value for key, value in flat
        if any(identifier in key.split(".")[-1].split("[")[0].lower() for identifier in potential_identifiers)
    ]
    #Remove any non-integer values from obs_count
    obs_count = [x for x in obs_count if isinstance(x, int) and not isinstance(x, bool)]
    
    if len(obs_count) > 0:
        #If we found any potential observation counts, check if they are within the expected range
        assert any(abs(n - expected) <= tolerance for n in obs_count), (
            f"No reported count near expected {expected} (tolerance {tolerance}) "
            f"found anywhere in the output. Integers present: {sorted(set(obs_count))}"
        )
        return

    # Level 2 (revised): infer count from the structure of an observations-like array
    candidate_arrays = list(_find_observation_arrays(agent_output))
    if candidate_arrays:
        lengths = [len(arr) for _, arr in candidate_arrays]
        assert any(abs(n - expected) <= tolerance for n in lengths), (
            f"No observations-like array with length near {expected} "
            f"(tolerance {tolerance}) found. Array lengths found: {lengths}"
        )
        return

    # If neither level 1 nor level 2 yielded a plausible observation count, fail the test 
    pytest.fail(f"Could not locate any observation count evidence in the output" )

test_observation_count(agent_output, flat, reference_groups)
    

In [ ]:




def test_observation_count(agent_output, flat, reference_groups):
    expected = len(reference_groups)
    tolerance = max(1, round(0.15 * expected))
    print(f"Expected observation count: {expected}")
    print(f"Tolerance: {tolerance}")

    #Level 1:
    #Start by searching for any keys in the output that contain potential observation count identifiers
    potential_identifiers = [
        'obs_count', 'observation_count', 'num_observations',
        'number_of_observations', 'num_obs', 'total_observations',
        'observations_total', 'observations_count', 'obs_total',
        'obs_num', 'obs_number', 'obs_count_total', 'obs_count_num',
        'obs_count_number',
    ]    
    #Check for any keys in the output that contain any of the potential identifiers
    obs_count = [
        value for key, value in flat
        if any(identifier in key.split(".")[-1].split("[")[0].lower() for identifier in potential_identifiers)
    ]
    #Remove any non-integer values from obs_count
    obs_count = [x for x in obs_count if isinstance(x, int) and not isinstance(x, bool)]
    
    if len(obs_count) > 0:
        #If we found any potential observation counts, check if they are within the expected range
        assert any(abs(n - expected) <= tolerance for n in obs_count), (
            f"No reported count near expected {expected} (tolerance {tolerance}) "
            f"found anywhere in the output. Integers present: {sorted(set(obs_count))}"
        )
    else: 
        #Otherwise, search for any integers in the output that are within the tolerance of the expected count
        print("No potential observation count identifiers found in the output. Searching for any integers in the output that are within the tolerance of the expected count.")
        ints_found = [v for _, v in flat if isinstance(v, int) and not isinstance(v, bool)]
        assert any(abs(n - expected) <= tolerance for n in ints_found), (
            f"No reported count near expected {expected} (tolerance {tolerance}) "
            f"found anywhere in the output. Integers present: {sorted(set(ints_found))}"
        )

test_observation_count(agent_output, flat, reference_groups)
    

Expected observation count: 14
Tolerance: 2


In [100]:


print([key.split(".")[-1].split("[")[0].lower() for key, value in flat])

['query_description', 'target', 'ra_deg', 'dec_deg', 'search_radius_arcmin', 'public', 'science', 'bands', 'bands', 'spatial_resolution_arcsec_max', 'num_observations', 'member_ous_uid', 'project_code', 'bands', 'spatial_resolution_arcsec', 'all_spatial_resolutions_arcsec', 'target_names', 's_ra_deg', 's_dec_deg', 'num_spectral_windows', 'member_ous_uid', 'project_code', 'bands', 'spatial_resolution_arcsec', 'all_spatial_resolutions_arcsec', 'target_names', 's_ra_deg', 's_dec_deg', 'num_spectral_windows', 'member_ous_uid', 'project_code', 'bands', 'spatial_resolution_arcsec', 'all_spatial_resolutions_arcsec', 'target_names', 's_ra_deg', 's_dec_deg', 'num_spectral_windows', 'member_ous_uid', 'project_code', 'bands', 'spatial_resolution_arcsec', 'all_spatial_resolutions_arcsec', 'target_names', 's_ra_deg', 's_dec_deg', 'num_spectral_windows', 'member_ous_uid', 'project_code', 'bands', 'spatial_resolution_arcsec', 'all_spatial_resolutions_arcsec', 'target_names', 's_ra_deg', 's_dec_deg', 

In [74]:
output_flat

[('query_description',
  'ALMA archive search for public, science observations near HH 212, Band 6 or 7, spatial_resolution < 1 arcsec'),
 ('target', 'HH 212'),
 ('resolved_coordinates.ra_deg', 85.96420833),
 ('resolved_coordinates.dec_deg', -1.04808333),
 ('search_radius_arcmin', 1.0),
 ('filters.public', True),
 ('filters.science', True),
 ('filters.bands[0]', 6),
 ('filters.bands[1]', 7),
 ('filters.spatial_resolution_arcsec_max', 1.0),
 ('num_observations', 14),
 ('observations[0].member_ous_uid', 'uid://A001/X1273/X1f9'),
 ('observations[0].project_code', '2017.1.00044.S'),
 ('observations[0].bands[0]', 7),
 ('observations[0].spatial_resolution_arcsec', 0.0316734361630697),
 ('observations[0].all_spatial_resolutions_arcsec[0]', 0.0316734361630697),
 ('observations[0].target_names[0]', 'HH_212'),
 ('observations[0].s_ra_deg', 85.96421125000545),
 ('observations[0].s_dec_deg', -1.0481019444447288),
 ('observations[0].num_spectral_windows', 4),
 ('observations[1].member_ous_uid', 'ui

In [75]:
output.keys()

dict_keys(['query_description', 'target', 'resolved_coordinates', 'search_radius_arcmin', 'filters', 'num_observations', 'observations'])

## Check DL table names

In [ ]:
# List all tables Data Lab actually hosts, matching common survey name fragments
result = qc.query(sql="""
    SELECT table_name FROM tap_schema.tables
    WHERE table_name ILIKE '%sdss%' OR table_name ILIKE '%des_%'
       OR table_name ILIKE '%ls_dr%' OR table_name ILIKE '%gaia%'
       OR table_name ILIKE '%2mass%' OR table_name ILIKE '%wise%'
""", fmt="pandas")
print(result)

                     table_name
0                allwise.source
1              des_dr2.coverage
2                  des_dr2.flux
3                   des_dr2.mag
4                  des_dr2.main
..                          ...
853         mydb.desi_qsos_3354
854      mydb.desi_bifrost_5870
855       mydb.des_dr2_d56_6818
856  mydb.wfst_nep_ls_dr10_4224
857      mydb.desi0412data_6320

[858 rows x 1 columns]


In [4]:
print(qc.schema('nsc_dr2.object'))


Schema: nsc_dr2
 Table: object

     Column Name   Description
     -----------   -----------
           ndetr   Number of detections in r
              ra   Right ascension (J2000.0) of object, in degrees
             dec   Declination (J2000.0) of object, in degrees
            glon   Galactic Longitude
            glat   Galactic Latitude
            elon   Ecliptic Longitude
            elat   Ecliptic Latitude
           raerr   Uncertainty of RA
          decerr   Uncertainty in DEC
            pmra   Proper motion in RA*cos(DEC)
           pmdec   Proper motion in DEC
         pmraerr   Uncertainty in PMRA
        pmdecerr   Uncertainty in PMDEC
             mjd   Mean Modified Julian Date
        deltamjd   Range of Modified Julian Date
            umag   Weighted-average u-band magnitude
            urms   RMS scatter in u from multiple measurements of this object
            uerr   Uncertainty in mean u-band magnitude
          uasemi   u-band profile RMS along major axis, i

In [11]:

SEARCH_RADIUS_ARCSEC = 30.0
coord = SkyCoord.from_name("NGC 1068")
# ALMA: no band/resolution filter, since none was requested 
alma = Alma()
tbl = alma.query_region(coord, radius=SEARCH_RADIUS_ARCSEC * u.arcsec, public=True, science=True)

groups = defaultdict(list)
for row in tbl:
    groups[str(row["member_ous_uid"])].append(row)

alma_observations = []
for mous_uid, rows in groups.items():
    bands = sorted({int(b) for r in rows for b in str(r["band_list"]).split()})
    alma_observations.append({
        "member_ous_uid": mous_uid,
        "project_code": rows[0]["proposal_id"],
        "band": bands,
    })

#DL
ra_deg, dec_deg = coord.ra.deg, coord.dec.deg
radius_deg = SEARCH_RADIUS_ARCSEC / 3600.0


nsc_query = f"""
    SELECT COUNT(*) as n FROM nsc_dr2.object
    WHERE q3c_radial_query(ra, dec, {ra_deg}, {dec_deg}, {radius_deg})
"""
nsc_result = qc.query(sql=nsc_query, fmt="pandas")
nsc_n_sources = int(nsc_result["n"].iloc[0])

output = {
    "target": "NGC 1068",
    "ra_deg": ra_deg,
    "dec_deg": dec_deg,
    "alma": {
        "n_observations": len(alma_observations),
        "observations": alma_observations,
    },
    "datalab": {
        "nsc_dr2_n_sources": nsc_n_sources,
        "nsc_dr2_has_coverage": nsc_n_sources > 0,
    },
}

In [12]:
output

{'target': 'NGC 1068',
 'ra_deg': np.float64(40.66962153),
 'dec_deg': np.float64(-0.01329436),
 'alma': {'n_observations': 107,
  'observations': [{'member_ous_uid': 'uid://A001/X1354/X5a2',
    'project_code': np.str_('2018.1.01684.S'),
    'band': [3]},
   {'member_ous_uid': 'uid://A001/X1354/X5a0',
    'project_code': np.str_('2018.1.01684.S'),
    'band': [3]},
   {'member_ous_uid': 'uid://A001/X1354/X5a4',
    'project_code': np.str_('2018.1.01684.S'),
    'band': [3]},
   {'member_ous_uid': 'uid://A001/X145/X302',
    'project_code': np.str_('2013.1.00188.S'),
    'band': [7]},
   {'member_ous_uid': 'uid://A001/X87d/X497',
    'project_code': np.str_('2016.1.00023.S'),
    'band': [6]},
   {'member_ous_uid': 'uid://A001/X145/X306',
    'project_code': np.str_('2013.1.00188.S'),
    'band': [6]},
   {'member_ous_uid': 'uid://A001/X145/X30a',
    'project_code': np.str_('2013.1.00188.S'),
    'band': [6]},
   {'member_ous_uid': 'uid://A001/X122/X4dc',
    'project_code': np.str_('

In [10]:
coord = SkyCoord.from_name("NGC 1068")
coord

<SkyCoord (ICRS): (ra, dec) in deg
    (40.66962153, -0.01329436)>

In [15]:

# To get image cutouts, you'll need the VO-based SIA package, and define which SIA service to use
from pyvo.dal import sia
import numpy as np

DEF_ACCESS_URL = "https://datalab.noirlab.edu/sia/des_dr1" # DES SIA service URL
svc = sia.SIAService(DEF_ACCESS_URL)

# NGC 288 (a globular cluster)
ra_deg, dec_deg = coord.ra, coord.dec


fov = 13/60  # image cutout field of view (in degrees; here 13 arminutes = 0.22 degrees)

imgTable = svc.search((ra_deg,dec_deg), (fov/np.cos(dec_deg*np.pi/180), fov), verbosity=2).to_table()  # uses declination correction

In [16]:
imgTable

assoc_id,access_url,access_format,access_estsize,dataproduct_type,dataproduct_subtype,calib_level,dataset_length,im_nsubarrays,im_naxes,im_naxis,im_pixtype,im_wcsaxes,im_scale,obs_title,obs_id,obs_creator_name,obs_collection,obs_creation_type,obs_creation_date,facility_name,instrument_name,obs_bandpass,obs_datasource,proposal_id,obs_publisher_did,obs_release_date,preview,target_name,target_class,s_ra,s_dec,s_fov,s_region,s_calib_status,s_resolution,em_min,em_max,em_resolution,em_res_power,t_min,t_max,t_exptime,t_resolution,o_ucd,o_unit,pol_states,object,airmass,project_code,magzero,prodtype,seeing,fwhm,fileset_id,exptime,photflag,proctype,date_obs,mjd_obs,elliptic
,,,bytes,,,,,,,,,,deg,,,,,,,,,,,,,,,,,deg,deg,deg,deg,,arcsec,m,m,m,,d,d,s,s,,,,,,,,,,,,,,,,,
object,object,object,int32,object,object,int64,int64,int32,int32,object,object,object,object,object,object,object,object,object,object,str1,object,object,object,object,object,object,object,object,object,float64,float64,float64,object,object,float64,float64,float64,float64,float64,float64,float64,float64,float64,object,object,object,object,object,object,object,object,object,object,object,object,object,object,object,object,object
des_dr1,"https://datalab.noirlab.edu/svc/cutout?col=des_dr1&siaRef=DES0242+0001_r2674p01_det.fits.fz&extn=1&POS=40.66962153,-0.01329436&SIZE=0.21666666666844334,0.21666666666666667",image/fits,86461,,,2,0,0,2,[10000 10000],,,[0.27],,des_dr1,,DES DR1,archival,,,DECam,,,,ivo://datalab.noirlab/vo#ivoa_des_dr1.siav1:ivo://datalab.noao/des_dr1/DES0242+0001_r2674p01_det.fits.fz,,"https://datalab.noirlab.edu/svc/cutout?col=des_dr1&siaRef=DES0242+0001_r2674p01_det.fits.fz&extn=1&POS=40.66962153,-0.01329436&SIZE=0.21666666666844334,0.21666666666666667&preview=true",,,40.6444,0.0166667,--,,,--,-2.98156e-39,2.98156e-39,--,--,--,--,--,--,,,,DES0242+0001,0,DES DR1,30,image,0,0,,1440,,Stack,,556545.3857379,0
des_dr1,"https://datalab.noirlab.edu/svc/cutout?col=des_dr1&siaRef=DES0242+0001_r2674p01_det.fits.fz&extn=2&POS=40.66962153,-0.01329436&SIZE=0.21666666666844334,0.21666666666666667",image/fits,86461,,,2,0,0,2,[10000 10000],,,[0.27],,des_dr1,,DES DR1,archival,,,DECam,,,,ivo://datalab.noirlab/vo#ivoa_des_dr1.siav1:ivo://datalab.noao/des_dr1/DES0242+0001_r2674p01_det.fits.fz,,"https://datalab.noirlab.edu/svc/cutout?col=des_dr1&siaRef=DES0242+0001_r2674p01_det.fits.fz&extn=2&POS=40.66962153,-0.01329436&SIZE=0.21666666666844334,0.21666666666666667&preview=true",,,40.6444,0.0166667,--,,,--,-3.13891e-43,3.13891e-43,--,--,--,--,--,--,,,,DES0242+0001,0,DES DR1,30,mask,0,0,,1440,,Stack,,556545.3857379,0
des_dr1,"https://datalab.noirlab.edu/svc/cutout?col=des_dr1&siaRef=DES0242+0001_r2674p01_det.fits.fz&extn=3&POS=40.66962153,-0.01329436&SIZE=0.21666666666844334,0.21666666666666667",image/fits,86461,,,2,0,0,2,[10000 10000],,,[0.27],,des_dr1,,DES DR1,archival,,,DECam,,,,ivo://datalab.noirlab/vo#ivoa_des_dr1.siav1:ivo://datalab.noao/des_dr1/DES0242+0001_r2674p01_det.fits.fz,,"https://datalab.noirlab.edu/svc/cutout?col=des_dr1&siaRef=DES0242+0001_r2674p01_det.fits.fz&extn=3&POS=40.66962153,-0.01329436&SIZE=0.21666666666844334,0.21666666666666667&preview=true",,,40.6444,0.0166667,--,,,--,-3.13891e-43,3.13891e-43,--,--,--,--,--,--,,,,DES0242+0001,0,DES DR1,30,weight,0,0,,1440,,Stack,,556545.3857379,0
des_dr1,"https://datalab.noirlab.edu/svc/cutout?col=des_dr1&siaRef=DES0242+0001_r2674p01_g.fits.fz&extn=1&POS=40.66962153,-0.01329436&SIZE=0.21666666666844334,0.21666666666666667",image/fits,164511,,,2,0,0,2,[10000 10000],,,[0.27],,des_dr1,,DES DR1,archival,,,DECam,g,,,ivo://datalab.noirlab/vo#ivoa_des_dr1.siav1:ivo://datalab.noao/des_dr1/DES0242+0001_r2674p01_g.fits.fz,,"https://datalab.noirlab.edu/svc/cutout?col=des_dr1&siaRef=DES0242+0001_r2674p01_g.fits.fz&extn=1&POS=40.66962153,-0.01329436&SIZE=0.21666666666844334,0.21666666666666667&preview=true",,,40.6444,0.0166667,--,,,--,3960.0,5480.0,--,--,--,--,--,--,,,,DES0242+0001,0,DES DR1,30,image,0,0,,450,,Stack,,556545.38434587,0
